# Milestone 6A: Apache Spark RDD Fundamentals for AML Detection

This notebook demonstrates low-level **Resilient Distributed Dataset (RDD)** transformations and actions using Apache Spark.
While higher-level DataFrames and Spark SQL provide optimization via Catalyst, understanding RDDs is fundamental for unstructured data processing and custom partition-level control.

### Key Concepts Demonstrated:
- **Ingestion & Parsing**: `sc.textFile()`, `filter()`, `map()`
- **Standardization**: Currency conversion mapping to USD reference
- **Key-Value Pair RDDs**: `flatMap()` to emit `(account, event)` tuples
- **Aggregation**: `reduceByKey()`, `mapValues()`
- **Sorting & Actions**: `sortByKey()`, `take()`, `count()`, `first()`, `saveAsTextFile()`
- **Set & Pair Operations**: `distinct()`, `groupByKey()`, `union()`, `intersection()`, `subtract()`, `aggregate()`


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_rdd_pipeline import (
    create_spark_session,
    run_rdd_pipeline,
    demonstrate_rdd_operations,
    parse_transaction_row,
    standardize_to_usd,
    EXCHANGE_RATES_TO_USD,
)

# Start local Spark session (or connect to Databricks cluster)
spark = create_spark_session('Databricks-M6A-RDD')
sc = spark.sparkContext
print('Spark Version:', sc.version)
print('Default Parallelism:', sc.defaultParallelism)


## 1. Raw Text Ingestion and Header Filtering
We read the untouched CSV file as raw lines of text without automated schema inference.
We use the `.first()` action to extract the header, then `.filter()` transformation to drop it.


In [ ]:
csv_path = '../data/raw/HI-Small_Trans.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/processed/processed_df.csv'

raw_lines = sc.textFile(csv_path)
header = raw_lines.first()  # Action: Retrieve first element
print('CSV Header Line:', header)

# Transformation: filter out header line
data_lines = raw_lines.filter(lambda line: line != header)
print('Sample record:', data_lines.take(1)[0])


## 2. Parsing, Currency Standardization, and Filtering (Transformations)
We parse CSV text into namedtuples and apply the exchange rate mapping `standardize_to_usd`.


In [ ]:
# Broadcast exchange rates for zero-overhead worker lookups
rates_bc = sc.broadcast(EXCHANGE_RATES_TO_USD)

# Map: line -> Transaction
parsed_rdd = data_lines.map(parse_transaction_row)

# Map: currency standardization to USD
standardized_rdd = parsed_rdd.map(lambda txn: standardize_to_usd(txn, rates_bc.value))

# Filter: keep valid transactions
valid_rdd = standardized_rdd.filter(
    lambda t: t.amount_paid_usd > 0 and t.amount_received_usd > 0 and t.from_account != t.to_account
).cache()  # Persist for reuse

print('Total Valid Transactions:', valid_rdd.count())


## 3. Emitting Per-Account Events with `flatMap`
Every transaction generates two directed events: an outflow for the sender and an inflow for the receiver.


In [ ]:
account_events = valid_rdd.flatMap(lambda txn: [
    (txn.from_account, {'dir': 'out', 'amt': txn.amount_paid_usd, 'peer': txn.to_account, 'fraud': txn.is_laundering}),
    (txn.to_account, {'dir': 'in', 'amt': txn.amount_received_usd, 'peer': txn.from_account, 'fraud': txn.is_laundering}),
])

print('Sample account events:')
for item in account_events.take(4):
    print(item)


## 4. Aggregating Account Features via `reduceByKey` and Sorting via `sortByKey`
We aggregate in/out degrees, USD volumes, and counterparties, then sort by a composite risk heuristic.


In [ ]:
results = run_rdd_pipeline(sc, csv_path)
print('Total unique accounts aggregated:', results['total_accounts'])
print('Ground-truth label counts:', results['label_counts'])
print('\nTop 5 Riskiest Accounts by RDD heuristic:')
for score, acct, feats in results['top_20'][:5]:
    print(f'Account: {acct} | Score: {score:.4f} | In-Deg: {feats["in_degree"]} | Out-Deg: {feats["out_degree"]} | Fan-In Ratio: {feats["fan_in_ratio"]:.2f}')


## 5. Additional RDD Primitives (`distinct`, `groupByKey`, `aggregate`)
Showcasing set and grouping operations across the transaction graph.


In [ ]:
demonstrate_rdd_operations(sc, csv_path)
spark.stop()
